In [2]:
from datasets import load_dataset

/home/xujiaming/xujiaming/anaconda3/envs/test_flash/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = load_dataset("/home/xujiaming/xujiaming/models/LongBench", "qasper", split="test",trust_remote_code=True)

In [7]:
data[0]['context']

"10pt\n1.10pt\n[ Characterizing Political Fake News in Twitter by its Meta-DataJulio Amador Díaz LópezAxel Oehmichen Miguel Molina-Solana( j.amador, axelfrancois.oehmichen11, mmolinas@imperial.ac.uk ) Imperial College London This article presents a preliminary approach towards characterizing political fake news on Twitter through the analysis of their meta-data. In particular, we focus on more than 1.5M tweets collected on the day of the election of Donald Trump as 45th president of the United States of America. We use the meta-data embedded within those tweets in order to look for differences between tweets containing fake news and tweets not containing them. Specifically, we perform our analysis only on tweets that went viral, by studying proxies for users' exposure to the tweets, by characterizing accounts spreading fake news, and by looking at their polarization. We found significant differences on the distribution of followers, the number of URLs on tweets, and the verification of

In [69]:
prompt_format = "You are given a scientific article and a question. Answer the question as concisely as you can, using a single phrase or sentence if possible. If the question cannot be answered based on the information in the article, write \"unanswerable\". If the question is a yes/no question, answer \"yes\", \"no\", or \"unanswerable\". Do not provide any explanation.\n\nArticle: {context}\n\n Answer the question based on the above article as concisely as you can, using a single phrase or sentence if possible. If the question cannot be answered based on the information in the article, write \"unanswerable\". If the question is a yes/no question, answer \"yes\", \"no\", or \"unanswerable\". Do not provide any explanation.\n\nQuestion: {input}\n\nAnswer:"
prompt = prompt_format.format(**data[0])

In [74]:
len(prompt)

20581

In [78]:
q_pos = prompt.rfind("Question:")

In [79]:
q_pos = max(len(prompt) - 100, q_pos)

In [80]:
q_pos

20512

In [63]:
data[0]

{'input': 'Passage:\nVisit Grand Coulee Dam| Bureau of Reclamation\nVisit Grand Coulee Dam| Bureau of Reclamation\nContact Us\nVisit the Dam\nExplore the dam, take part in the D3 Geocache Challenge, view the Laser Light Show, and come inside the Visitor Center to experience the hands-on exhibits!\nThe visitor center is open daily (except Thanksgiving Day, December 25, and January 1) from 9:00 a.m. to 5:00 p.m, with extended hours between Memorial Day and September 30. During the summer season the visitor center is open until the laser light show, One River, Many Voices, ends. Show times vary, learn more >>\nQuestion:\nIn which country is the Grand Coulee Dam\nAnswer:\n',
 'context': 'Passage:\nAdam\'s apple\nThe laryngeal prominence (commonly referred to as Adam\'s apple), a feature of the human neck, is the lump or protrusion that is formed by the angle of the thyroid cartilage surrounding the larynx.\n\nStructure\n\nThe structure of the laryngeal prominence forms a bump under the ski

In [58]:
tot = 0
tot_len = 0
minn = 100000
maxx = 0
for i in data:
    maxx = max(maxx, i['length'])
    minn = min(minn, i['length']) 
    tot += 1
    tot_len += i['length']
print(tot_len/tot)
print('(',minn,',',maxx,')')

8209.295
( 1145 , 16633 )


In [8]:
from models.Mer_model import Mer_Model
from models.kv_cache import initialize_past_key_values
from models.utils import *
from typing import Optional
from lm_eval.api.model import LM
from lm_eval.api.instance import  Instance
import os

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
Spec_model_path = "/home/xujiaming/xujiaming/models/EAGLE3-LLaMA3.1-Instruct-8B"
Ori_model_path = "/share/public/public_models/Llama-3.1-8B-Instruct"
mer_model = Mer_Model.from_pretrained(
    Spec_model_path = Spec_model_path,
    Ori_model_path = Ori_model_path,
    torch_dtype = torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

class SpecKV_Model(LM):
    def __init__(self, model, batch_size: Optional[int] = None):
        super().__init__()
        self.model = model

    def loglikelihood(self, requests: list[Instance]) -> list[tuple[float, bool]]:
        pass


    def loglikelihood_rolling(self, requests: list[Instance]) -> list[tuple[float, bool]]:
        pass


    def generate_until(self, requests: list[Instance]) -> list[str]:
        pass

LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.16s/it]


In [10]:
lm_obj = SpecKV_Model(mer_model)

In [ ]:

task_manager = lm_eval.tasks.TaskManager()
results = lm_eval.simple_evaluate(
    model=lm_obj,
    tasks=["gsm8k"],
    num_fewshot=0,
    task_manager=task_manager,)

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 263504.98 examples/s]
Overwriting default num_fewshot of gsm8k from 5 to 0
100%|██████████| 1319/1319 [00:00<00:00, 3171.49it/s]


TypeError: 'NoneType' object is not iterable

In [12]:
isinstance(lm_obj, lm_eval.api.model.LM)

True